In [32]:
import pandas as pd
import numpy as np
import time
from datetime import date
import glob
import os

In [35]:
parent_directory = "/home/schmiedc/FMP_Docs/Projects/2024-11-25_Bioactives_Revision/create_MOA/"

In [36]:
### Broad institute drug repurposing database
Broad = pd.read_csv(parent_directory + "Repurposing_Hub_export.txt", sep='\t')
Broad = Broad.drop(columns = "Deprecated ID")

In [37]:
# reduce columns
Broad_cols = ['Name', 'MOA', 'Target', 'Disease Area',
       'Indication', 'Id',
       'SMILES', 'InChIKey']

Broad_Red = Broad.loc[:, Broad_cols].copy()

In [38]:
Broad_Red = Broad_Red.add_prefix('Broad_') ### adds Broad_ in front of all column names

In [39]:
# Replace "," with ";" in multi-entry columns
replace_cols = ['Broad_MOA', 'Broad_Target', 'Broad_Disease Area','Broad_Indication', 'Broad_Id','Broad_SMILES', 'Broad_InChIKey']

for col in replace_cols:
    Broad_Red[col] = Broad_Red[col].str.replace(", ", ";")

Broad_Red["Broad_Name"] = Broad_Red["Broad_Name"].str.upper()

In [40]:
key_file = pd.read_csv(parent_directory + "EUOS_pdid.csv")
EUopen_Compounds = pd.read_csv(parent_directory + "pd_export_02_2024_2464_compounds_standardized.csv")

In [41]:
Compounds_cols = ['pdid', 'name', 'smiles', 'inchi', 'inchikey', 'no. targets', 'cas', 'synonyms', 'mw']

EUopen_Compounds_Red = EUopen_Compounds.loc[:, Compounds_cols].copy()
EUopen_Compounds_EUOS = pd.merge(EUopen_Compounds_Red, key_file, on='pdid', how='inner')

In [42]:
EUopen_Compounds_EUOS.head()

,pdid,name,smiles,inchi,inchikey,no. targets,cas,synonyms,mw,EOS
0,PD000002,I-BRD9,CCn1cc(-c2cccc(C(F)(F)F)c2)c2sc(C(=N)NC3CCS(=O...,InChI=1S/C22H22F3N3O3S2/c1-2-28-12-17(13-4-3-5...,WRUWGLUCNBMGPS-UHFFFAOYSA-N,7.0,1714146-59-4,CS-5587;GSK602;I-BRD9;I-BRD9 (GSK602),497.11,EOS101163
1,PD000003,UNC0638,COc1cc2c(NC3CCN(C(C)C)CC3)nc(C3CCCCC3)nc2cc1OC...,InChI=1S/C30H47N5O2/c1-22(2)35-17-12-24(13-18-...,QOECJCJVIMVJGX-UHFFFAOYSA-N,6.0,1255580-76-7;1255517-77-1,UNC-0638;UNC0638 hydrate;UNC0638;UNC 0638,509.37,EOS101593
2,PD000005,UNC1215,O=C(c1ccc(C(=O)N2CCC(N3CCCC3)CC2)c(Nc2ccccc2)c...,InChI=1S/C32H43N5O2/c38-31(36-20-12-27(13-21-3...,PQOOIERVZAXHBP-UHFFFAOYSA-N,11.0,1415800-43-9,UNC1215;UNC 1215,529.34,EOS101154
3,PD000007,IOX1,O=C(O)c1ccc(O)c2ncccc12,InChI=1S/C10H7NO3/c12-8-4-3-7(10(13)14)6-2-1-5...,JGRPKOGHYBAVMW-UHFFFAOYSA-N,17.0,5852-78-8,IOX-1;IOX1;IOX 1;Hydroxyqunoline analog 1,189.04,EOS101601
4,PD000008,IOX2,O=C(O)CNC(=O)c1c(O)c2ccccc2n(Cc2ccccc2)c1=O,InChI=1S/C19H16N2O5/c22-15(23)10-20-18(25)16-1...,CAOSCCRYLYQBES-UHFFFAOYSA-N,4.0,931398-72-0,JICL38;IOX2;IOX 2,352.11,EOS101116


# combine Broad and EUOPEN 

In [43]:
len(Broad_Red.loc[Broad_Red["Broad_Name"].isin(EUopen_Compounds_EUOS.name)])

933

In [44]:
EUopen_Broad_Anno = EUopen_Compounds_EUOS.merge(Broad_Red, left_on = ['name'], right_on = ['Broad_Name'], how = "inner")

In [45]:
annot_columns = ['pdid', 'EOS', 'Broad_MOA']
data_annotations_moa = EUopen_Broad_Anno.loc[:, annot_columns]

data_annotations_moa['Broad_MOA'] = data_annotations_moa['Broad_MOA'].str.split(';')
data_annotations_moa_full = data_annotations_moa.explode('Broad_MOA', ignore_index=True)

value_counts_MOA = data_annotations_moa_full['Broad_MOA'].value_counts()
MOA_keep = value_counts_MOA[value_counts_MOA >= 2].index
annotations_moa_filtered = data_annotations_moa_full[data_annotations_moa_full['Broad_MOA'].isin(MOA_keep)]

annotations_moa_filtered = annotations_moa_filtered.rename(columns={'pdid': 'Metadata_pdid', 
                                    'EOS': 'Metadata_EOS', 
                                    'Broad_MOA': 'Metadata_Broad_MOA'
                                    })

In [ ]:
len(annotations_moa_filtered)

867

In [47]:
filename = parent_directory + str(date.today()) + "_MOA_Broad.csv"
annotations_moa_filtered.to_csv(filename, index = False)

In [50]:
Broad_Red.head()

,Broad_Name,Broad_MOA,Broad_Target,Broad_Disease Area,Broad_Indication,Broad_Id,Broad_SMILES,Broad_InChIKey
0,(R)-(-)-APOMORPHINE,dopamine receptor agonist,ADRA2A;ADRA2B;ADRA2C;CALY;DRD1;DRD2;DRD3;DRD4;...,neurology/psychiatry,Parkinson's Disease,BRD-K76022557-003-28-9;BRD-K76022557-003-02-7;...,CN1CCc2cccc-3c2[C@H]1Cc1ccc(O)c(O)c-31;CN1CCc2...,VMWNQDUVQKEIOC-CYBMUJFWSA-N;VMWNQDUVQKEIOC-CYB...
1,(R)-(-)-ROLIPRAM,phosphodiesterase inhibitor,PDE4A;PDE4B;PDE4C;PDE4D;PDE5A,NaN,NaN,BRD-K75516118-001-04-1;BRD-K75516118-001-04-1;...,COc1ccc(cc1OC1CCCC1)[C@@H]1CNC(=O)C1;COc1ccc(c...,HJORMJIFDVBMOB-LBPRGKRZSA-N;HJORMJIFDVBMOB-LBP...
2,(R)-BACLOFEN,benzodiazepine receptor agonist,GABBR1;GABBR2,NaN,NaN,BRD-K62353271-001-04-7;BRD-K62353271-001-04-7;...,NC[C@H](CC(O)=O)c1ccc(Cl)cc1;NC[C@H](CC(O)=O)c...,KPYSYYIEGFHWSV-QMMMGPOBSA-N;KPYSYYIEGFHWSV-QMM...
3,(S)-(+)-ROLIPRAM,phosphodiesterase inhibitor,PDE4B;PDE4D,NaN,NaN,BRD-K65856711-001-05-9;BRD-K65856711-001-05-9;...,COc1ccc(cc1OC1CCCC1)[C@H]1CNC(=O)C1;COc1ccc(cc...,HJORMJIFDVBMOB-GFCCVEGCSA-N;HJORMJIFDVBMOB-GFC...
4,"[SAR9,MET(O2)11]-SUBSTANCE-P",tachykinin antagonist,TACR1,NaN,NaN,BRD-K89787693-001-02-9,CC(C)C[C@H](NC(=O)CN(C)C(=O)[C@H](Cc1ccccc1)NC...,OUPXSLGGCPUZJJ-SARDKLJWSA-N


In [51]:
EUopen_Broad_Anno = EUopen_Compounds_EUOS.merge(Broad_Red, left_on = ['smiles'], right_on = ['Broad_SMILES'], how = "inner")

annot_columns = ['pdid', 'EOS', 'Broad_MOA']
data_annotations_moa = EUopen_Broad_Anno.loc[:, annot_columns]

data_annotations_moa['Broad_MOA'] = data_annotations_moa['Broad_MOA'].str.split(';')
data_annotations_moa_full = data_annotations_moa.explode('Broad_MOA', ignore_index=True)

value_counts_MOA = data_annotations_moa_full['Broad_MOA'].value_counts()
MOA_keep = value_counts_MOA[value_counts_MOA >= 2].index
annotations_moa_filtered = data_annotations_moa_full[data_annotations_moa_full['Broad_MOA'].isin(MOA_keep)]

annotations_moa_filtered = annotations_moa_filtered.rename(columns={'pdid': 'Metadata_pdid', 
                                    'EOS': 'Metadata_EOS', 
                                    'Broad_MOA': 'Metadata_Broad_MOA'
                                    })


len(annotations_moa_filtered)

2